In [ ]:
# Lecture 8 — Transformations vs Actions
# Spark 101 · Module 3 · Lecture 8
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.

In [ ]:
# Load data — this is a transformation. Returns a DataFrame. No job runs.
df = spark.read.table("samples.nyctaxi.trips")

In [ ]:
# Chain four more transformations. Still no jobs. Check the Spark UI — empty.
from pyspark.sql.functions import col

short_trips = df.filter(col("trip_distance") < 5)
grouped = short_trips.groupBy("pickup_zip").count()
sorted_df = grouped.orderBy(col("count").desc())
top_five = sorted_df.limit(5)

# This cell finishes instantly. Zero work has been done.
print("Plan built — no jobs triggered yet.")

In [ ]:
# An action. NOW the whole plan runs — one Job appears in the Spark UI.
top_five.show()

In [ ]:
# A second action on the same DataFrame.
# The whole plan runs AGAIN — Spark doesn't remember the first run.
# Check the UI: two identical Jobs, doubled runtime.
print(f"Row count: {top_five.count()}")

In [ ]:
# .cache() would fix this (Module 7) — but for now, just observe:
# transformations queue, actions fire, and each action is a full re-run.

In [ ]:
# More transformations — still lazy, still zero jobs.
from pyspark.sql.functions import col
enriched = short_trips.withColumn("per_mile", col("fare_amount") / col("trip_distance"))
projected = enriched.select("pickup_zip", "fare_amount", "per_mile")
print("more plan built — still no jobs")

In [ ]:
# Several ACTIONS — each one triggers its own Job.
print("first:", top_five.first())
print("take 3:", top_five.take(3))
print("count:", top_five.count())

In [ ]:
# write is an action too — it runs the whole plan and saves output.
top_five.write.mode("overwrite").parquet("file:/tmp/spark101_l08/top_five")
print("written — that was an action")